# Проверка API распознавания вина

Запустите сервис из корня проекта через `docker compose up --build -d` или `uv run uvicorn app.main:app --host 127.0.0.1 --port 8810`. Выполняйте ячейки по порядку. Ключ OpenRouter нужен сервису, но не ноутбуку. Каждый запрос `/v1/predict` обращается к модели и расходует токены.


In [ ]:
import base64
import csv
from html import escape
from pathlib import Path
from time import perf_counter

import httpx
from IPython.display import HTML, display


In [ ]:
BASE_URL = "http://127.0.0.1:8810"
client = httpx.Client(base_url=BASE_URL, timeout=90.0, trust_env=False)


def show(response: httpx.Response):
    print(f"{response.request.method} {response.request.url} → {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    display(payload)
    response.raise_for_status()
    return payload


In [ ]:
show(client.get("/health"))


## Фото без slug

Источник — `C:\Users\eryom\GitHub\vinishko\data\test\test.csv`. Ноутбук берёт только строки, где `slug` пустой, и проверяет наличие фото в `images/`. Сейчас таких строк 41. По умолчанию выбраны 12 разных фото из этого набора, включая все три JPEG и WebP из разных частей списка. Установите `RUN_ALL = True`, чтобы отправить все 41 фото. Эталонных ответов для них здесь нет: результаты нужно оценивать по изображениям.


In [ ]:
TEST_DIR = Path(r"C:\Users\eryom\GitHub\vinishko\data\test")
RUN_ALL = False

with (TEST_DIR / "test.csv").open(encoding="utf-8-sig", newline="") as source:
    rows = list(csv.DictReader(source))
no_slug_rows = [row for row in rows if not row["slug"].strip()]
if not no_slug_rows:
    raise ValueError("В test.csv нет строк с пустым slug")
missing = [row["image_filename"] for row in no_slug_rows
           if not (TEST_DIR / "images" / row["image_filename"]).is_file()]
if missing:
    raise FileNotFoundError(f"Отсутствуют фото: {missing}")

default_names = (
    "test_000107.jpg",
    "test_000108.webp",
    "test_000111.webp",
    "test_000115.webp",
    "test_000119.webp",
    "test_000123.webp",
    "test_000126.jpg",
    "test_000130.webp",
    "test_000133.jpg",
    "test_000138.webp",
    "test_000142.webp",
    "test_000147.webp",
)
if RUN_ALL:
    cases = no_slug_rows
else:
    selected = set(default_names)
    cases = [row for row in no_slug_rows if row["image_filename"] in selected]
    if len(cases) != len(default_names):
        raise ValueError("Часть тестовых фото больше не имеет пустой slug или отсутствует в test.csv")
if len({row["image_filename"] for row in cases}) != len(cases):
    raise ValueError("В наборе повторяются имена фото")

print(f"Строк без slug: {len(no_slug_rows)}. Выбрано фото: {len(cases)}")
for number, row in enumerate(cases, 1):
    print(f"{number:02d}. {row['image_filename']}")


## Посмотрите фото

Измените `CASE_NUMBER` и повторно выполните ячейку, чтобы посмотреть любой выбранный кейс. Формат для предпросмотра определяется по сигнатуре файла.


In [ ]:
CASE_NUMBER = 1


def preview_case(number: int):
    if not 1 <= number <= len(cases):
        raise IndexError(f"Номер должен быть от 1 до {len(cases)}")
    path = TEST_DIR / "images" / cases[number - 1]["image_filename"]
    data = path.read_bytes()
    if data.startswith(b"\xff\xd8\xff"):
        media_type = "image/jpeg"
    elif data.startswith(b"\x89PNG\r\n\x1a\n"):
        media_type = "image/png"
    elif data[:4] == b"RIFF" and data[8:12] == b"WEBP":
        media_type = "image/webp"
    elif data.startswith((b"GIF87a", b"GIF89a")):
        media_type = "image/gif"
    else:
        raise ValueError(f"Неподдерживаемый формат: {path}")
    print(f"{number:02d}. {path.name} ({len(data):,} байт)")
    data_url = base64.b64encode(data).decode("ascii")
    display(HTML(f'<img src="data:{media_type};base64,{data_url}" style="max-width:360px">'))


preview_case(CASE_NUMBER)


## Отправьте выбранные фото

Запросы выполняются последовательно, по одному на фото. Таблица покажет ответ и время каждого вызова. Ошибка одного запроса не остановит остальные. Значение `Не удалось определить` — допустимый успешный ответ по отдельному полю.


In [ ]:
results = []
for number, row in enumerate(cases, 1):
    path = TEST_DIR / "images" / row["image_filename"]
    started = perf_counter()
    try:
        with path.open("rb") as image_file:
            response = client.post(
                "/v1/predict",
                files={"image": (path.name, image_file, "application/octet-stream")},
            )
        status = response.status_code
        try:
            payload = response.json()
        except ValueError:
            payload = response.text
        if status == 200 and isinstance(payload, dict):
            category = payload.get("category", "")
            brand = payload.get("brand", "")
            error = "" if set(payload) == {"category", "brand"} else "Неожиданный формат ответа"
        else:
            category = ""
            brand = ""
            error = payload.get("detail", "") if isinstance(payload, dict) else str(payload)
    except httpx.RequestError as exc:
        status = "сеть"
        category = ""
        brand = ""
        error = str(exc)
    elapsed = perf_counter() - started
    results.append({
        "№": number,
        "Фото": path.name,
        "HTTP": status,
        "Категория": category,
        "Винодельня": brand,
        "Время, с": f"{elapsed:.2f}",
        "Ошибка": error,
    })
    print(f"{number:02d}/{len(cases)} {path.name}: HTTP {status}, {elapsed:.2f} с")

columns = ("№", "Фото", "HTTP", "Категория", "Винодельня", "Время, с", "Ошибка")
header = "".join(f"<th>{escape(name)}</th>" for name in columns)
body = "".join(
    "<tr>" + "".join(f"<td>{escape(str(result[name]))}</td>" for name in columns) + "</tr>"
    for result in results
)
display(HTML(
    "<table style='border-collapse:collapse'>"
    f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"
))
unknown = "Не удалось определить"
print(f"Успешных ответов: {sum(r['HTTP'] == 200 for r in results)}/{len(results)}")
print(f"Отказов по категории: {sum(r['Категория'] == unknown for r in results)}")
print(f"Отказов по винодельне: {sum(r['Винодельня'] == unknown for r in results)}")
